# Silver Layer - Data Transformation

This notebook transforms raw bronze layer data into clean, validated silver layer tables ready for analytics.

## Data Model Overview

**Fact Table:**
- `factsalestable` - Sales transactions with foreign keys to customers, products, and dates

**Dimension Tables:**
- `dimcustomertable` - Customer information with geography reference
- `dimdatetable` - Date dimension with calendar attributes
- `dimregiontable` - Geography/region information
- `dimproducttable` - Products with subcategory reference
- `dimproductsubcategorytable` - Product subcategories with category reference
- `dimproductcategorytable` - Product categories (top level)

## Transformation Steps

1. **Analyze & Profile** - Understand bronze table structures and relationships
2. **Validate Keys** - Check referential integrity and key uniqueness
3. **Data Quality** - Remove nulls, duplicates, and fix data types
4. **Create Joins** - Build denormalized views for analytics
5. **Save to Silver** - Persist transformed tables to workspace.silver

In [0]:
# Read and analyze the fact sales table
fact_sales = spark.table("workspace.bronze.factsalestable")

print("=" * 80)
print("FACT TABLE: factsalestable")
print("=" * 80)
print(f"\nTotal rows: {fact_sales.count():,}")
print("\nSchema:")
fact_sales.printSchema()
print("\nSample data (first 10 rows):")
display(fact_sales.limit(10))

# Check for nulls in key columns
print("\nNull counts in key columns:")
from pyspark.sql.functions import col, sum as spark_sum, when

key_columns = ['ProductKey', 'OrderDateKey', 'CustomerKey']
for col_name in key_columns:
    null_count = fact_sales.filter(col(col_name).isNull()).count()
    print(f"  {col_name}: {null_count:,} nulls")

FACT TABLE: factsalestable

Total rows: 114,390

Schema:
root
 |-- ProductKey: string (nullable = true)
 |-- OrderDateKey: string (nullable = true)
 |-- CustomerKey: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- OrderNumber: string (nullable = true)
 |-- OrderQuantity: string (nullable = true)
 |-- List_Price: string (nullable = true)
 |-- Product_Cost: string (nullable = true)


Sample data (first 10 rows):


ProductKey,OrderDateKey,CustomerKey,Gender,OrderNumber,OrderQuantity,List_Price,Product_Cost
344,20050722,11000,null,20061722,22,3399.99,1912.1543999999999
353,20070722,11000,null,20081722,22,2319.9899999999998,1265.6195
485,20070722,11000,null,20081722,22,21.98,8.2204999999999995
530,20071104,11000,null,20082104,4,4.99,1.8663000000000001
214,20071104,11000,null,20082104,4,34.99,13.0863
488,20071104,11000,null,20082104,4,53.99,41.572299999999998
573,20071104,11000,null,20082104,4,2384.0700000000002,1481.9378999999999
541,20071104,11000,null,20082104,4,28.99,10.8423
350,20050718,11001,null,20061719,18,3374.99,1898.0944
477,20070720,11001,null,20081721,20,4.99,1.8663000000000001



Null counts in key columns:
  ProductKey: 0 nulls
  OrderDateKey: 0 nulls
  CustomerKey: 0 nulls


In [0]:
# Read and analyze all dimension tables

dim_tables = {
    'dimcustomertable': ['CustomerKey', 'GeographyKey'],
    'dimdatetable': ['DateKey'],
    'dimregiontable': ['GeographyKey'],
    'dimproducttable': ['ProductKey', 'ProductSubcategoryKey'],
    'dimproductsubcategorytable': ['ProductSubcategoryKey', 'ProductCategoryKey'],
    'dimproductcategorytable': ['ProductCategoryKey']
}

for table_name, key_columns in dim_tables.items():
    df = spark.table(f"workspace.bronze.{table_name}")
    
    print("\n" + "=" * 80)
    print(f"DIMENSION TABLE: {table_name}")
    print("=" * 80)
    print(f"\nTotal rows: {df.count():,}")
    print("\nSchema:")
    df.printSchema()
    print(f"\nSample data (first 5 rows):")
    display(df.limit(5))
    
    # Check for nulls in key columns
    print("\nNull counts in key columns:")
    for col_name in key_columns:
        null_count = df.filter(col(col_name).isNull()).count()
        print(f"  {col_name}: {null_count:,} nulls")


DIMENSION TABLE: dimcustomertable

Total rows: 9,999

Schema:
root
 |-- GeographyKey: string (nullable = true)
 |-- MaritalStatus: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- CustomerKey: string (nullable = true)
 |-- CustomerName: string (nullable = true)


Sample data (first 5 rows):


GeographyKey,MaritalStatus,Gender,CustomerKey,CustomerName
612,M,M,11254,Customer1
612,M,M,11255,Customer2
612,M,M,11282,Customer3
612,M,M,11321,Customer4
612,M,M,11633,Customer5



Null counts in key columns:
  CustomerKey: 0 nulls
  GeographyKey: 0 nulls

DIMENSION TABLE: dimdatetable

Total rows: 1,188

Schema:
root
 |-- DateKey: string (nullable = true)
 |-- FullDateAlternateKey: string (nullable = true)
 |-- EnglishMonthName: string (nullable = true)
 |-- CalendarYear: string (nullable = true)


Sample data (first 5 rows):


DateKey,FullDateAlternateKey,EnglishMonthName,CalendarYear
20060101,1/1/06 0:00,January,2006
20060102,1/2/06 0:00,January,2006
20060103,1/3/06 0:00,January,2006
20060104,1/4/06 0:00,January,2006
20060105,1/5/06 0:00,January,2006



Null counts in key columns:
  DateKey: 0 nulls

DIMENSION TABLE: dimregiontable

Total rows: 655

Schema:
root
 |-- GeographyKey: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Country: string (nullable = true)


Sample data (first 5 rows):


...


Null counts in key columns:
  ProductKey: 0 nulls
  ProductSubcategoryKey: 0 nulls

DIMENSION TABLE: dimproductsubcategorytable

Total rows: 40

Schema:
root
 |-- ProductSubcategoryKey: string (nullable = true)
 |-- ProductSubcategoryName: string (nullable = true)
 |-- ProductCategoryKey: string (nullable = true)


Sample data (first 5 rows):


ProductSubcategoryKey,ProductSubcategoryName,ProductCategoryKey
1,Mountain Bikes,1
2,Road Bikes,1
3,Touring Bikes,1
4,Handlebars,2
5,Bottom Brackets,2



Null counts in key columns:
  ProductSubcategoryKey: 0 nulls
  ProductCategoryKey: 0 nulls

DIMENSION TABLE: dimproductcategorytable

Total rows: 5

Schema:
root
 |-- ProductCategoryKey: string (nullable = true)
 |-- ProductCategoryName: string (nullable = true)


Sample data (first 5 rows):


ProductCategoryKey,ProductCategoryName
1,Bikes
2,Components
3,Clothing
4,Accessories
5,Food



Null counts in key columns:
  ProductCategoryKey: 0 nulls


In [0]:
# Check primary key uniqueness for all dimension tables

print("=" * 80)
print("PRIMARY KEY UNIQUENESS VALIDATION")
print("=" * 80)

# Check each dimension table's primary key
key_checks = {
    'dimcustomertable': 'CustomerKey',
    'dimdatetable': 'DateKey',
    'dimregiontable': 'GeographyKey',
    'dimproducttable': 'ProductKey',
    'dimproductsubcategorytable': 'ProductSubcategoryKey',
    'dimproductcategorytable': 'ProductCategoryKey'
}

for table_name, key_col in key_checks.items():
    df = spark.table(f"workspace.bronze.{table_name}")
    total_rows = df.count()
    distinct_keys = df.select(key_col).distinct().count()
    duplicates = total_rows - distinct_keys
    
    status = "✓ UNIQUE" if duplicates == 0 else f"✗ DUPLICATES FOUND: {duplicates}"
    print(f"\n{table_name}.{key_col}:")
    print(f"  Total rows: {total_rows:,}")
    print(f"  Distinct keys: {distinct_keys:,}")
    print(f"  Status: {status}")

print("\n" + "=" * 80)

PRIMARY KEY UNIQUENESS VALIDATION

dimcustomertable.CustomerKey:
  Total rows: 9,999
  Distinct keys: 9,999
  Status: ✓ UNIQUE

dimdatetable.DateKey:
  Total rows: 1,188
  Distinct keys: 1,188
  Status: ✓ UNIQUE

dimregiontable.GeographyKey:
  Total rows: 655
  Distinct keys: 655
  Status: ✓ UNIQUE

dimproducttable.ProductKey:
  Total rows: 400
  Distinct keys: 400
  Status: ✓ UNIQUE

dimproductsubcategorytable.ProductSubcategoryKey:
  Total rows: 40
  Distinct keys: 40
  Status: ✓ UNIQUE

dimproductcategorytable.ProductCategoryKey:
  Total rows: 5
  Distinct keys: 5
  Status: ✓ UNIQUE



In [0]:
# Check for orphaned foreign keys (foreign keys in fact table that don't exist in dimension tables)

print("=" * 80)
print("REFERENTIAL INTEGRITY VALIDATION - Checking for Orphaned Records")
print("=" * 80)

# Load tables
fact_sales = spark.table("workspace.bronze.factsalestable")
dim_customer = spark.table("workspace.bronze.dimcustomertable")
dim_date = spark.table("workspace.bronze.dimdatetable")
dim_product = spark.table("workspace.bronze.dimproducttable")

# Check 1: Orphaned CustomerKey in fact table
orphaned_customers = fact_sales.select("CustomerKey").distinct() \
    .join(dim_customer.select("CustomerKey"), "CustomerKey", "left_anti")
orphaned_customer_count = orphaned_customers.count()

print(f"\n1. CustomerKey Orphans:")
print(f"   Orphaned records: {orphaned_customer_count:,}")
if orphaned_customer_count > 0:
    print(f"   Sample orphaned keys:")
    display(orphaned_customers.limit(10))

# Check 2: Orphaned OrderDateKey in fact table
orphaned_dates = fact_sales.select("OrderDateKey").distinct() \
    .join(dim_date.select("DateKey"), fact_sales.OrderDateKey == dim_date.DateKey, "left_anti")
orphaned_date_count = orphaned_dates.count()

print(f"\n2. OrderDateKey Orphans:")
print(f"   Orphaned records: {orphaned_date_count:,}")
if orphaned_date_count > 0:
    print(f"   Sample orphaned keys:")
    display(orphaned_dates.limit(10))

# Check 3: Orphaned ProductKey in fact table
orphaned_products = fact_sales.select("ProductKey").distinct() \
    .join(dim_product.select("ProductKey"), "ProductKey", "left_anti")
orphaned_product_count = orphaned_products.count()

print(f"\n3. ProductKey Orphans:")
print(f"   Orphaned records: {orphaned_product_count:,}")
if orphaned_product_count > 0:
    print(f"   Sample orphaned keys:")
    display(orphaned_products.limit(10))

# Check 4: Orphaned GeographyKey in customer dimension
dim_region = spark.table("workspace.bronze.dimregiontable")
orphaned_geography = dim_customer.select("GeographyKey").distinct() \
    .join(dim_region.select("GeographyKey"), "GeographyKey", "left_anti")
orphaned_geography_count = orphaned_geography.count()

print(f"\n4. GeographyKey Orphans (in dimcustomertable):")
print(f"   Orphaned records: {orphaned_geography_count:,}")
if orphaned_geography_count > 0:
    print(f"   Sample orphaned keys:")
    display(orphaned_geography.limit(10))

# Check 5: Orphaned ProductSubcategoryKey in product dimension
dim_subcat = spark.table("workspace.bronze.dimproductsubcategorytable")
orphaned_subcat = dim_product.select("ProductSubcategoryKey").distinct() \
    .join(dim_subcat.select("ProductSubcategoryKey"), "ProductSubcategoryKey", "left_anti")
orphaned_subcat_count = orphaned_subcat.count()

print(f"\n5. ProductSubcategoryKey Orphans (in dimproducttable):")
print(f"   Orphaned records: {orphaned_subcat_count:,}")
if orphaned_subcat_count > 0:
    print(f"   Sample orphaned keys:")
    display(orphaned_subcat.limit(10))

# Check 6: Orphaned ProductCategoryKey in subcategory dimension
dim_cat = spark.table("workspace.bronze.dimproductcategorytable")
orphaned_cat = dim_subcat.select("ProductCategoryKey").distinct() \
    .join(dim_cat.select("ProductCategoryKey"), "ProductCategoryKey", "left_anti")
orphaned_cat_count = orphaned_cat.count()

print(f"\n6. ProductCategoryKey Orphans (in dimproductsubcategorytable):")
print(f"   Orphaned records: {orphaned_cat_count:,}")
if orphaned_cat_count > 0:
    print(f"   Sample orphaned keys:")
    display(orphaned_cat.limit(10))

print("\n" + "=" * 80)

# Summary
total_issues = orphaned_customer_count + orphaned_date_count + orphaned_product_count + \
               orphaned_geography_count + orphaned_subcat_count + orphaned_cat_count

if total_issues == 0:
    print("\n✓ VALIDATION PASSED: No orphaned foreign keys found!")
else:
    print(f"\n✗ VALIDATION FAILED: {total_issues:,} orphaned foreign key records found")
    print("   These records will need to be filtered out or fixed in the silver layer.")

REFERENTIAL INTEGRITY VALIDATION - Checking for Orphaned Records

1. CustomerKey Orphans:
   Orphaned records: 7,655
   Sample orphaned keys:


CustomerKey
11001
11008
11026
11052
11072
11078
11144
11227
11266
11267



2. OrderDateKey Orphans:
   Orphaned records: 0

3. ProductKey Orphans:
   Orphaned records: 0

4. GeographyKey Orphans (in dimcustomertable):
   Orphaned records: 0

5. ProductSubcategoryKey Orphans (in dimproducttable):
   Orphaned records: 0

6. ProductCategoryKey Orphans (in dimproductsubcategorytable):
   Orphaned records: 0


✗ VALIDATION FAILED: 7,655 orphaned foreign key records found
   These records will need to be filtered out or fixed in the silver layer.


In [0]:
# Transform the fact sales table: fix data types and filter orphaned records

from pyspark.sql.functions import col, to_date

print("=" * 80)
print("TRANSFORMING FACT TABLE: factsalestable")
print("=" * 80)

# Load fact and dimension tables
fact_sales_raw = spark.table("workspace.bronze.factsalestable")
dim_customer = spark.table("workspace.bronze.dimcustomertable")

print(f"\nOriginal fact table rows: {fact_sales_raw.count():,}")

# Step 1: Filter out orphaned CustomerKeys (keep only valid customers)
fact_sales_valid = fact_sales_raw.join(
    dim_customer.select("CustomerKey"),
    "CustomerKey",
    "inner"  # Inner join ensures we only keep valid customers
)

print(f"After filtering orphaned customers: {fact_sales_valid.count():,} rows")

# Step 2: Convert data types from strings to appropriate types
fact_sales_transformed = fact_sales_valid.select(
    col("ProductKey").cast("int").alias("ProductKey"),
    col("OrderDateKey").cast("int").alias("OrderDateKey"),
    col("CustomerKey").cast("int").alias("CustomerKey"),
    col("Gender").cast("string").alias("Gender"),
    col("OrderNumber").cast("string").alias("OrderNumber"),
    col("OrderQuantity").cast("int").alias("OrderQuantity"),
    col("List_Price").cast("decimal(10,2)").alias("List_Price"),
    col("Product_Cost").cast("decimal(10,2)").alias("Product_Cost")
)

# Step 3: Add calculated columns
fact_sales_transformed = fact_sales_transformed.withColumn(
    "Revenue",
    col("List_Price") * col("OrderQuantity")
).withColumn(
    "Cost",
    col("Product_Cost") * col("OrderQuantity")
).withColumn(
    "Profit",
    (col("List_Price") - col("Product_Cost")) * col("OrderQuantity")
)

print("\n✓ Transformations applied:")
print("  - Filtered orphaned customer records")
print("  - Converted ProductKey, OrderDateKey, CustomerKey to INT")
print("  - Converted OrderQuantity to INT")
print("  - Converted List_Price, Product_Cost to DECIMAL(10,2)")
print("  - Added calculated columns: Revenue, Cost, Profit")

print("\nTransformed schema:")
fact_sales_transformed.printSchema()

print("\nSample transformed data:")
display(fact_sales_transformed.limit(10))

print("\n" + "=" * 80)

TRANSFORMING FACT TABLE: factsalestable

Original fact table rows: 114,390
After filtering orphaned customers: 63,738 rows

✓ Transformations applied:
  - Filtered orphaned customer records
  - Converted ProductKey, OrderDateKey, CustomerKey to INT
  - Converted OrderQuantity to INT
  - Converted List_Price, Product_Cost to DECIMAL(10,2)
  - Added calculated columns: Revenue, Cost, Profit

Transformed schema:
root
 |-- ProductKey: integer (nullable = true)
 |-- OrderDateKey: integer (nullable = true)
 |-- CustomerKey: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- OrderNumber: string (nullable = true)
 |-- OrderQuantity: integer (nullable = true)
 |-- List_Price: decimal(10,2) (nullable = true)
 |-- Product_Cost: decimal(10,2) (nullable = true)
 |-- Revenue: decimal(21,2) (nullable = true)
 |-- Cost: decimal(21,2) (nullable = true)
 |-- Profit: decimal(22,2) (nullable = true)


Sample transformed data:


ProductKey,OrderDateKey,CustomerKey,Gender,OrderNumber,OrderQuantity,List_Price,Product_Cost,Revenue,Cost,Profit
344,20050722,11000,null,20061722,22,3399.99,1912.15,74799.78,42067.30,32732.48
353,20070722,11000,null,20081722,22,2319.99,1265.62,51039.78,27843.64,23196.14
485,20070722,11000,null,20081722,22,21.98,8.22,483.56,180.84,302.72
530,20071104,11000,null,20082104,4,4.99,1.87,19.96,7.48,12.48
214,20071104,11000,null,20082104,4,34.99,13.09,139.96,52.36,87.60
488,20071104,11000,null,20082104,4,53.99,41.57,215.96,166.28,49.68
573,20071104,11000,null,20082104,4,2384.07,1481.94,9536.28,5927.76,3608.52
541,20071104,11000,null,20082104,4,28.99,10.84,115.96,43.36,72.60
346,20050710,11002,null,20061712,10,3399.99,1912.15,33999.90,19121.50,14878.40
359,20070704,11002,null,20081706,4,2294.99,1251.98,9179.96,5007.92,4172.04


In [0]:
# Transform all dimension tables: fix data types

print("=" * 80)
print("TRANSFORMING DIMENSION TABLES")
print("=" * 80)

# 1. Transform Customer Dimension
print("\n1. Transforming dimcustomertable...")
dim_customer_raw = spark.table("workspace.bronze.dimcustomertable")
dim_customer_transformed = dim_customer_raw.select(
    col("CustomerKey").cast("int").alias("CustomerKey"),
    col("GeographyKey").cast("int").alias("GeographyKey"),
    col("CustomerName").cast("string").alias("CustomerName"),
    col("Gender").cast("string").alias("Gender"),
    col("MaritalStatus").cast("string").alias("MaritalStatus")
)
print(f"   Rows: {dim_customer_transformed.count():,}")

# 2. Transform Date Dimension
print("\n2. Transforming dimdatetable...")
dim_date_raw = spark.table("workspace.bronze.dimdatetable")
dim_date_transformed = dim_date_raw.select(
    col("DateKey").cast("int").alias("DateKey"),
    to_date(col("FullDateAlternateKey"), "M/d/yy H:mm").alias("FullDate"),
    col("EnglishMonthName").cast("string").alias("MonthName"),
    col("CalendarYear").cast("int").alias("CalendarYear")
)
print(f"   Rows: {dim_date_transformed.count():,}")

# 3. Transform Region Dimension
print("\n3. Transforming dimregiontable...")
dim_region_raw = spark.table("workspace.bronze.dimregiontable")
dim_region_transformed = dim_region_raw.select(
    col("GeographyKey").cast("int").alias("GeographyKey"),
    col("City").cast("string").alias("City"),
    col("Region").cast("string").alias("Region"),
    col("Country").cast("string").alias("Country")
)
print(f"   Rows: {dim_region_transformed.count():,}")

# 4. Transform Product Dimension
print("\n4. Transforming dimproducttable...")
dim_product_raw = spark.table("workspace.bronze.dimproducttable")
dim_product_transformed = dim_product_raw.select(
    col("ProductKey").cast("int").alias("ProductKey"),
    col("ProductSubcategoryKey").cast("int").alias("ProductSubcategoryKey"),
    col("Product_Name").cast("string").alias("Product_Name"),
    col("Color").cast("string").alias("Color"),
    col("StandardCost").cast("decimal(10,2)").alias("StandardCost"),
    col("ListPrice").cast("decimal(10,2)").alias("ListPrice")
)
print(f"   Rows: {dim_product_transformed.count():,}")

# 5. Transform Product Subcategory Dimension
print("\n5. Transforming dimproductsubcategorytable...")
dim_subcat_raw = spark.table("workspace.bronze.dimproductsubcategorytable")
dim_subcat_transformed = dim_subcat_raw.select(
    col("ProductSubcategoryKey").cast("int").alias("ProductSubcategoryKey"),
    col("ProductSubcategoryName").cast("string").alias("ProductSubcategoryName"),
    col("ProductCategoryKey").cast("int").alias("ProductCategoryKey")
)
print(f"   Rows: {dim_subcat_transformed.count():,}")

# 6. Transform Product Category Dimension
print("\n6. Transforming dimproductcategorytable...")
dim_cat_raw = spark.table("workspace.bronze.dimproductcategorytable")
dim_cat_transformed = dim_cat_raw.select(
    col("ProductCategoryKey").cast("int").alias("ProductCategoryKey"),
    col("ProductCategoryName").cast("string").alias("ProductCategoryName")
)
print(f"   Rows: {dim_cat_transformed.count():,}")

print("\n" + "=" * 80)
print("\n✓ All dimension tables transformed successfully!")
print("  - All key columns converted to INT")
print("  - Date columns converted to DATE type")
print("  - Price/cost columns converted to DECIMAL(10,2)")

TRANSFORMING DIMENSION TABLES

1. Transforming dimcustomertable...
   Rows: 9,999

2. Transforming dimdatetable...
   Rows: 1,188

3. Transforming dimregiontable...
   Rows: 655

4. Transforming dimproducttable...
   Rows: 400

5. Transforming dimproductsubcategorytable...
   Rows: 40

6. Transforming dimproductcategorytable...
   Rows: 5


✓ All dimension tables transformed successfully!
  - All key columns converted to INT
  - Date columns converted to DATE type
  - Price/cost columns converted to DECIMAL(10,2)


In [0]:
# Create a comprehensive denormalized view with all business context

print("=" * 80)
print("CREATING DENORMALIZED SALES FACT VIEW")
print("=" * 80)
print("\nBusiness Purpose: Create a complete sales analysis table by joining the fact table")
print("with all dimension tables to provide full business context for each transaction.")
print("\nJoin Logic:")
print("  1. Fact -> Customer: Get customer demographics (gender, marital status, name)")
print("  2. Customer -> Region: Get geographic information (city, region, country)")
print("  3. Fact -> Date: Get date attributes (full date, month, year)")
print("  4. Fact -> Product: Get product details (name, color, costs, prices)")
print("  5. Product -> Subcategory: Get product subcategory (e.g., Mountain Bikes)")
print("  6. Subcategory -> Category: Get top-level category (e.g., Bikes)")
print("\n" + "=" * 80)

# Join all tables together
fact_denormalized = fact_sales_transformed \
    .join(dim_customer_transformed, "CustomerKey", "inner") \
    .join(dim_region_transformed, "GeographyKey", "inner") \
    .join(dim_date_transformed, fact_sales_transformed.OrderDateKey == dim_date_transformed.DateKey, "inner") \
    .join(dim_product_transformed, "ProductKey", "inner") \
    .join(dim_subcat_transformed, "ProductSubcategoryKey", "inner") \
    .join(dim_cat_transformed, "ProductCategoryKey", "inner") \
    .select(
        # Transaction details
        col("OrderNumber"),
        col("OrderDateKey"),
        col("FullDate").alias("OrderDate"),
        col("MonthName"),
        col("CalendarYear"),
        
        # Customer details
        col("CustomerKey"),
        col("CustomerName"),
        fact_sales_transformed.Gender.alias("CustomerGender"),
        col("MaritalStatus"),
        
        # Geography details
        col("GeographyKey"),
        col("City"),
        col("Region"),
        col("Country"),
        
        # Product details
        col("ProductKey"),
        col("Product_Name"),
        col("Color"),
        col("ProductSubcategoryKey"),
        col("ProductSubcategoryName"),
        col("ProductCategoryKey"),
        col("ProductCategoryName"),
        
        # Transaction metrics
        col("OrderQuantity"),
        col("List_Price"),
        col("Product_Cost"),
        col("StandardCost"),
        col("ListPrice"),
        col("Revenue"),
        col("Cost"),
        col("Profit")
    )

row_count = fact_denormalized.count()
print(f"\n✓ Denormalized view created with {row_count:,} rows")
print(f"   Columns: {len(fact_denormalized.columns)}")

print("\nSchema:")
fact_denormalized.printSchema()

print("\nSample denormalized data (first 5 rows):")
display(fact_denormalized.limit(5))

print("\n" + "=" * 80)

CREATING DENORMALIZED SALES FACT VIEW

Business Purpose: Create a complete sales analysis table by joining the fact table
with all dimension tables to provide full business context for each transaction.

Join Logic:
  1. Fact -> Customer: Get customer demographics (gender, marital status, name)
  2. Customer -> Region: Get geographic information (city, region, country)
  3. Fact -> Date: Get date attributes (full date, month, year)
  4. Fact -> Product: Get product details (name, color, costs, prices)
  5. Product -> Subcategory: Get product subcategory (e.g., Mountain Bikes)
  6. Subcategory -> Category: Get top-level category (e.g., Bikes)


✓ Denormalized view created with 63,738 rows
   Columns: 28

Schema:
root
 |-- OrderNumber: string (nullable = true)
 |-- OrderDateKey: integer (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- MonthName: string (nullable = true)
 |-- CalendarYear: integer (nullable = true)
 |-- CustomerKey: integer (nullable = true)
 |-- CustomerName

OrderNumber,OrderDateKey,OrderDate,MonthName,CalendarYear,CustomerKey,CustomerName,CustomerGender,MaritalStatus,GeographyKey,City,Region,Country,ProductKey,Product_Name,Color,ProductSubcategoryKey,ProductSubcategoryName,ProductCategoryKey,ProductCategoryName,OrderQuantity,List_Price,Product_Cost,StandardCost,ListPrice,Revenue,Cost,Profit
20061722,20050722,2005-07-22,July,2005,11000,Customer337,null,M,26,Rockhampton,Queensland,Australia,344,Product 89,Silver,1,Mountain Bikes,1,Bikes,22,3399.99,1912.15,1912.15,3399.99,74799.78,42067.30,32732.48
20081722,20070722,2007-07-22,July,2007,11000,Customer337,null,M,26,Rockhampton,Queensland,Australia,353,Product 94,Silver,1,Mountain Bikes,1,Bikes,22,2319.99,1265.62,1265.62,2319.99,51039.78,27843.64,23196.14
20081722,20070722,2007-07-22,July,2007,11000,Customer337,null,M,26,Rockhampton,Queensland,Australia,485,Product 163,NA,30,Fenders,4,Accessories,22,21.98,8.22,8.22,21.98,483.56,180.84,302.72
20082104,20071104,2007-11-04,November,2007,11000,Customer337,null,M,26,Rockhampton,Queensland,Australia,530,Product 208,NA,37,Tires and Tubes,4,Accessories,4,4.99,1.87,1.87,4.99,19.96,7.48,12.48
20082104,20071104,2007-11-04,November,2007,11000,Customer337,null,M,26,Rockhampton,Queensland,Australia,214,Product 3,Red,31,Helmets,4,Accessories,4,34.99,13.09,13.09,34.99,139.96,52.36,87.60


In [0]:
# Save all transformed tables to workspace.silver schema

print("=" * 80)
print("SAVING TRANSFORMED TABLES TO SILVER LAYER")
print("=" * 80)

# Create silver schema if it doesn't exist
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")
print("\n✓ Schema workspace.silver ready")

# Save fact table
print("\n1. Saving fact_sales_silver...")
fact_sales_transformed.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.fact_sales")
print(f"   ✓ Saved {fact_sales_transformed.count():,} rows")

# Save dimension tables
print("\n2. Saving dim_customer_silver...")
dim_customer_transformed.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.dim_customer")
print(f"   ✓ Saved {dim_customer_transformed.count():,} rows")

print("\n3. Saving dim_date_silver...")
dim_date_transformed.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.dim_date")
print(f"   ✓ Saved {dim_date_transformed.count():,} rows")

print("\n4. Saving dim_region_silver...")
dim_region_transformed.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.dim_region")
print(f"   ✓ Saved {dim_region_transformed.count():,} rows")

print("\n5. Saving dim_product_silver...")
dim_product_transformed.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.dim_product")
print(f"   ✓ Saved {dim_product_transformed.count():,} rows")

print("\n6. Saving dim_product_subcategory_silver...")
dim_subcat_transformed.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.dim_product_subcategory")
print(f"   ✓ Saved {dim_subcat_transformed.count():,} rows")

print("\n7. Saving dim_product_category_silver...")
dim_cat_transformed.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.dim_product_category")
print(f"   ✓ Saved {dim_cat_transformed.count():,} rows")

# Save denormalized view
print("\n8. Saving fact_sales_denormalized...")
fact_denormalized.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.fact_sales_denormalized")
print(f"   ✓ Saved {fact_denormalized.count():,} rows")

print("\n" + "=" * 80)
print("\n✓✓✓ ALL TABLES SUCCESSFULLY SAVED TO SILVER LAYER! ✓✓✓")
print("\nSilver layer tables ready for analytics:")
print("  - workspace.silver.fact_sales (cleaned sales transactions)")
print("  - workspace.silver.dim_customer (customer dimension)")
print("  - workspace.silver.dim_date (date dimension)")
print("  - workspace.silver.dim_region (geography dimension)")
print("  - workspace.silver.dim_product (product dimension)")
print("  - workspace.silver.dim_product_subcategory (product subcategory)")
print("  - workspace.silver.dim_product_category (product category)")
print("  - workspace.silver.fact_sales_denormalized (complete denormalized view)")
print("\n" + "=" * 80)

SAVING TRANSFORMED TABLES TO SILVER LAYER

✓ Schema workspace.silver ready

1. Saving fact_sales_silver...
   ✓ Saved 63,738 rows

2. Saving dim_customer_silver...
   ✓ Saved 9,999 rows

3. Saving dim_date_silver...
   ✓ Saved 1,188 rows

4. Saving dim_region_silver...
   ✓ Saved 655 rows

5. Saving dim_product_silver...
   ✓ Saved 400 rows

6. Saving dim_product_subcategory_silver...
   ✓ Saved 40 rows

7. Saving dim_product_category_silver...
   ✓ Saved 5 rows

8. Saving fact_sales_denormalized...
   ✓ Saved 63,738 rows


✓✓✓ ALL TABLES SUCCESSFULLY SAVED TO SILVER LAYER! ✓✓✓

Silver layer tables ready for analytics:
  - workspace.silver.fact_sales (cleaned sales transactions)
  - workspace.silver.dim_customer (customer dimension)
  - workspace.silver.dim_date (date dimension)
  - workspace.silver.dim_region (geography dimension)
  - workspace.silver.dim_product (product dimension)
  - workspace.silver.dim_product_subcategory (product subcategory)
  - workspace.silver.dim_product_cat

# Transformation Summary

## ✅ Completed Tasks

### 1. Data Analysis & Profiling
- Analyzed all 7 bronze tables (1 fact, 6 dimensions)
- Validated schemas and identified data types
- Confirmed 114,390 sales transactions in fact table
- No null values found in key columns

### 2. Data Quality Validation
- **Primary Keys**: All dimension tables have unique primary keys ✓
- **Foreign Keys**: Found and filtered 7,655 orphaned CustomerKeys
- Final cleaned fact table: **63,738 valid transactions**

### 3. Data Type Transformations
- Converted all key columns from STRING to INT
- Converted price/cost columns to DECIMAL(10,2)
- Converted date columns to proper DATE type
- Added calculated metrics: Revenue, Cost, Profit

### 4. Denormalized View Created
- Built comprehensive sales analysis table
- Joined fact table with all 6 dimensions
- Includes complete business context:
  * Customer demographics & geography
  * Product hierarchy (category → subcategory → product)
  * Date attributes
  * Transaction metrics

### 5. Silver Layer Tables

All tables saved to **workspace.silver**:

| Table | Rows | Description |
| --- | --- | --- |
| fact_sales | 63,738 | Cleaned sales transactions |
| dim_customer | 9,999 | Customer dimension |
| dim_date | 1,188 | Date dimension |
| dim_region | 655 | Geography dimension |
| dim_product | 400 | Product dimension |
| dim_product_subcategory | 40 | Product subcategories |
| dim_product_category | 5 | Product categories |
| fact_sales_denormalized | 63,738 | Complete denormalized view |

## 🎯 Ready for Analytics

The silver layer data is now:
- ✓ Clean (no orphaned records)
- ✓ Validated (referential integrity checked)
- ✓ Properly typed (correct data types)
- ✓ Enriched (calculated metrics added)
- ✓ Denormalized (ready for analysis)

## 📊 Suggested Next Steps

1. **Gold Layer**: Create aggregated fact tables for specific business questions
2. **BI Dashboards**: Build visualizations using the denormalized view
3. **Data Quality Monitoring**: Set up regular checks for orphaned records
4. **Performance**: Add partitioning on OrderDate for large-scale queries